# schedule paper execution v1

验证 `src.schedule` 新增的 paper/sim 执行管线：从 dashboard export 和 data registry 读取输入，生成 execution intent，使用 paper broker 执行，并写出 ledger 与 paper execution result。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import src.data as data_api
from src.schedule import PaperExecutionPipelineConfig, run_paper_execution_once

CARD = Path('research/schedule_paper_execution_v1')
CARD.mkdir(parents=True, exist_ok=True)
LOCKS = CARD / 'locks'
DASHBOARD_PATH = CARD / 'dashboard_export.json'
REGISTRY_PATH = CARD / 'data' / 'data_registry.json'
LEDGER_PATH = CARD / 'intent_ledger.json'
EXECUTION_PATH = CARD / 'paper_execution_result.json'
SUMMARY_PATH = CARD / 'analysis_summary.json'

In [2]:
raw = pd.DataFrame(
    [
        {'date': '2026-01-02', 'instrument': 'SOXL', 'close': 10.0, 'volume': 100},
        {'date': '2026-01-05', 'instrument': 'SOXL', 'close': 12.0, 'volume': 120},
    ]
)
calendar = pd.DatetimeIndex(pd.to_datetime(['2026-01-02', '2026-01-05']), name='date')
snapshot = data_api.build_canonical_snapshot(
    raw,
    name='schedule_paper_execution_unit_daily',
    universe=('SOXL',),
    calendar=calendar,
    calendar_path='data/qlib/soxl/calendars/day.txt',
    source_paths=('provider:unit',),
    start='2026-01-01',
    end='2026-01-05',
    as_of='2026-01-05',
)
data_api.publish_canonical_snapshot(
    snapshot,
    output_dir=CARD / 'data' / 'snapshots' / 'schedule_paper_execution_unit_daily' / '2026-01-05',
    dataset_key='daily.schedule_paper_execution_unit',
    registry_path=REGISTRY_PATH,
    published_at='2026-01-05T21:00:00Z',
)

dashboard_payload = {
    'schema_version': 'dashboard.model_matrix.v1',
    'artifact_id': 'schedule_paper_execution_unit_dashboard',
    'artifact_type': 'dashboard_export',
    'top_models': [
        {
            'rank': 1,
            'model': 'ridge_mse__all_features',
            'base_model': 'ridge_mse',
            'feature_set': 'all_features',
            'avg_position': 0.5,
            'stability_score': 4.0,
            'max_drawdown_pct': 8.0,
        }
    ],
    'objective': {
        'promotion_scores': [
            {
                'rank': 1,
                'model': 'ridge_mse__all_features',
                'promotion_score': 12.0,
                'promotion_decision': 'promote_candidate',
                'promotion_blockers': '',
            }
        ]
    },
}
DASHBOARD_PATH.write_text(json.dumps(dashboard_payload, ensure_ascii=False, indent=2), encoding='utf-8')

DASHBOARD_PATH, REGISTRY_PATH

(PosixPath('research/schedule_paper_execution_v1/dashboard_export.json'),
 PosixPath('research/schedule_paper_execution_v1/data/data_registry.json'))

In [3]:
events: list[str] = []
config = PaperExecutionPipelineConfig(
    name='schedule_paper_execution_v1',
    dashboard_export_path=DASHBOARD_PATH,
    data_registry_path=REGISTRY_PATH,
    dataset_key='daily.schedule_paper_execution_unit',
    instrument='SOXL',
    intent_ledger_path=LEDGER_PATH,
    paper_execution_result_path=EXECUTION_PATH,
    initial_cash=1000.0,
    notify_details=True,
)
result = run_paper_execution_once(config, lock_dir=LOCKS, notify=events.append)
result

JobResult(name='schedule_paper_execution_v1', status='success', payload={'schema_version': 'schedule.paper_execution.v1', 'status': 'paper_executed', 'live_broker_enabled': False, 'mode': 'paper', 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。', 'instrument': 'SOXL', 'dataset_key': 'daily.schedule_paper_execution_unit', 'dashboard_export_path': 'research/schedule_paper_execution_v1/dashboard_export.json', 'data_registry_path': 'research/schedule_paper_execution_v1/data/data_registry.json', 'intent_ledger_path': 'research/schedule_paper_execution_v1/intent_ledger.json', 'paper_execution_result_path': 'research/schedule_paper_execution_v1/paper_execution_result.json', 'source_model': 'ridge_mse__all_features', 'source_artifact': 'schedule_paper_execution_unit_dashboard', 'target_position': 0.5, 'execution_gate': 'paper_review', 'quote': {'schema_version': 'api.market_quote.v1', 'instrument': 'SOXL', 'date': '2026-01-05', 'price': 12.0, 'field': 'close', 'data_mode': 'paper', 'sourc

In [4]:
ledger = json.loads(LEDGER_PATH.read_text(encoding='utf-8'))
execution = json.loads(EXECUTION_PATH.read_text(encoding='utf-8'))
summary = {
    'schema_version': 'research.schedule_paper_execution_v1.summary',
    'status': result.status,
    'pipeline_schema_version': result.payload['schema_version'],
    'live_broker_enabled': result.payload['live_broker_enabled'],
    'mode': result.payload['mode'],
    'instrument': result.payload['instrument'],
    'dataset_key': result.payload['dataset_key'],
    'source_model': result.payload['source_model'],
    'execution_gate': result.payload['execution_gate'],
    'target_position': result.payload['target_position'],
    'quote_date': result.payload['quote']['date'],
    'quote_price': result.payload['quote']['price'],
    'shares': result.payload['snapshot']['shares'],
    'realized_position': result.payload['snapshot']['position'],
    'intent_ledger': str(LEDGER_PATH),
    'paper_execution_result': str(EXECUTION_PATH),
    'notifications': events,
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

assert result.status == 'success'
assert summary['pipeline_schema_version'] == 'schedule.paper_execution.v1'
assert summary['live_broker_enabled'] is False
assert summary['mode'] == 'paper'
assert summary['execution_gate'] == 'paper_review'
assert summary['quote_date'] == '2026-01-05'
assert summary['quote_price'] == 12.0
assert summary['shares'] == 41
assert ledger['schema_version'] == 'api.intent_ledger.v1'
assert execution['schema_version'] == 'api.paper_execution_result.v1'

summary

{'schema_version': 'research.schedule_paper_execution_v1.summary',
 'status': 'success',
 'pipeline_schema_version': 'schedule.paper_execution.v1',
 'live_broker_enabled': False,
 'mode': 'paper',
 'instrument': 'SOXL',
 'dataset_key': 'daily.schedule_paper_execution_unit',
 'source_model': 'ridge_mse__all_features',
 'execution_gate': 'paper_review',
 'target_position': 0.5,
 'quote_date': '2026-01-05',
 'quote_price': 12.0,
 'shares': 41,
 'realized_position': 0.492,
 'intent_ledger': 'research/schedule_paper_execution_v1/intent_ledger.json',
 'paper_execution_result': 'research/schedule_paper_execution_v1/paper_execution_result.json',
 'notifications': ['schedule_paper_execution_v1:success | mode=paper | gate=paper_review | model=ridge_mse__all_features | target=0.5 | quote=SOXL@2026-01-05/12 | shares=41 | ledger=research/schedule_paper_execution_v1/intent_ledger.json']}

## 中文结论

- 做了什么：新增并验证 `src.schedule.run_paper_execution_once`，把 dashboard export、data registry、paper market quote、execution intent、paper broker、intent ledger 和 paper execution result 串成可调度管线。
- 核心指标：测试样例使用 `SOXL`、`ridge_mse__all_features`、目标仓位 `0.5`、报价日期 `2026-01-05`、价格 `12.0`、初始现金 `1000.0`，最终 paper shares 为 `41`，实际仓位约 `0.492`。
- regime 差异：本实验只验证执行链路，不重新评估 regime；regime 风险仍来自 dashboard payload 中已经固化的 analysis/objective gate。
- 是否晋升：可作为 production paper/sim 执行调度接口使用；不晋升为 live broker 接口，`live_broker_enabled=False`，v1 仍只允许 `paper/sim`。
- 风险：当前 paper broker 未处理真实成交、部分成交、盘中流动性和外部 broker 状态；真实 broker 适配必须另起架构扩展决策。
- 下一步：把 production notebook 的手写 paper replay 改为调用该 schedule 管线，并让 production health 继续检查 `intent_ledger` 与 `paper_execution_result`。